# Figure 2: Private reputation and directional inertia

Run from this notebook directory. Saved data reproduce 12 reputation matrices, four depth panels, and one combined figure. Set the corresponding recomputation flag to generate simulation data. The full depth scan contains 196 simulations. Requires NumPy, pandas, Matplotlib, Numba, and IPython.

In [1]:
from pathlib import Path
from typing import Tuple
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from numba import njit
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / "data" / "figure2"
FIGURES = ROOT / "figures" / "figure2"
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
RECOMPUTE_MATRICES = False
RECOMPUTE_DEPTH_SCAN = False

NORMS = ("SC", "SH", "SJ", "SS")
NORM_PARAMETERS = {"SC": (1., 0.), "SH": (0., 0.), "SJ": (0., 1.), "SS": (1., 1.)}
NORM_NAMES = {"SC": "Scoring", "SH": "Shunning", "SJ": "Stern Judging", "SS": "Simple Standing"}
TARGETS = ("DISC", "ALLC", "ALLD")
REGIMES = ("without_inertia", "one_step", "multi_step")
REGIME_NAMES = {"without_inertia": "Without inertia", "one_step": "One-step inertia", "multi_step": "Multi-step inertia"}
MULTISTEP_DEPTHS = {"SC": (2, 0), "SH": (3, 0), "SJ": (4, 3), "SS": (1, 2)}
MATRIX_PARAMETERS = dict(n_disc=30, n_allc=30, n_alld=30, e1=.02, e2=.02, sweeps=1000, seed=2027)
DEPTH_PARAMETERS = dict(norms=list(NORMS), tolerance_depths=list(range(7)), strictness_depths=list(range(7)),
    n_disc=50, n_allc=50, n_alld=50, e1=.02, e2=.02, sweeps_total=1000., sweeps_burnin=800.,
    sample_interval_sweeps=5., n_reps=1, seed=2027)
METRICS = ("g_DISC", "g_ALLC", "g_ALLD", "disagreement_DISC", "disagreement_ALLC", "disagreement_ALLD", "warning_fraction", "suspicious_fraction")
SUMMARY_METRICS = METRICS + ("mean_disagreement", "cooperator_defector_separation", "intermediate_fraction")

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.linewidth": .7,
    "pdf.fonttype": 42, "svg.fonttype": "none", "figure.dpi": 140})


## Reputation process

Records use direct-reset transitions. A positive assessment clears any warning; a negative assessment clears any recovery state. Binary readout is sampled independently whenever a record is consulted. DISC observers are sampled independently of donor and recipient; self-images are retained.

In [2]:
def build_sigma(alpha: float, beta: float, k1: int, k2: int) -> np.ndarray:
    sigma = np.empty(2 + k1 + k2, dtype=np.float64)
    sigma[0] = 1.0
    for m in range(1, k1 + 1):
        sigma[m] = alpha ** m
    b_index = k1 + 1
    sigma[b_index] = 0.0
    for n in range(1, k2 + 1):
        sigma[b_index + n] = 1.0 - beta ** n
    return sigma

def build_reset_transitions(k1: int, k2: int) -> Tuple[np.ndarray, np.ndarray]:
    n_states = 2 + k1 + k2
    b_index = k1 + 1
    positive = np.empty(n_states, dtype=np.int16)
    negative = np.empty(n_states, dtype=np.int16)
    positive[0] = 0
    negative[0] = 1 if k1 > 0 else b_index
    for m in range(1, k1 + 1):
        positive[m] = 0
        negative[m] = m + 1 if m < k1 else b_index
    positive[b_index] = b_index + 1 if k2 > 0 else 0
    negative[b_index] = b_index
    for n in range(1, k2 + 1):
        state = b_index + n
        positive[state] = state + 1 if n < k2 else 0
        negative[state] = b_index
    return (positive, negative)

@njit(cache=False)
def _simulate_condition(n_disc: int, n_allc: int, n_alld: int, p_norm: float, q_norm: float, e1: float, e2: float, sigma: np.ndarray, positive: np.ndarray, negative: np.ndarray, n_events: int, use_seed: bool, seed: int) -> np.ndarray:
    if use_seed:
        np.random.seed(seed)
    population_size = n_disc + n_allc + n_alld
    latent = np.zeros((n_disc, population_size), dtype=np.int16)
    for _ in range(n_events):
        donor = np.random.randint(0, population_size)
        recipient = np.random.randint(0, population_size - 1)
        if recipient >= donor:
            recipient += 1
        observer = np.random.randint(0, n_disc)
        if donor < n_disc:
            intended_cooperation = int(np.random.random() < sigma[latent[donor, recipient]])
        elif donor < n_disc + n_allc:
            intended_cooperation = 1
        else:
            intended_cooperation = 0
        if intended_cooperation == 1:
            realized_cooperation = int(np.random.random() >= e1)
        else:
            realized_cooperation = 0
        recipient_good = int(np.random.random() < sigma[latent[observer, recipient]])
        if realized_cooperation == 1:
            positive_probability = 1.0 if recipient_good == 1 else p_norm
        else:
            positive_probability = 0.0 if recipient_good == 1 else q_norm
        recorded_positive = int(np.random.random() < positive_probability)
        if np.random.random() < e2:
            recorded_positive = 1 - recorded_positive
        current = latent[observer, donor]
        latent[observer, donor] = positive[current] if recorded_positive == 1 else negative[current]
    return latent

@njit(cache=False)
def _binary_draw(probability: float) -> int:
    return 1 if np.random.random() < probability else 0

@njit(cache=False)
def _accumulate_statistics(latent: np.ndarray, sigma: np.ndarray, n_disc: int, n_allc: int, k1: int, accumulator: np.ndarray) -> None:
    population_size = latent.shape[1]
    starts = (0, n_disc, n_disc + n_allc)
    ends = (n_disc, n_disc + n_allc, population_size)
    for group in range(3):
        sum_target_goodness = 0.0
        sum_target_disagreement = 0.0
        n_targets = ends[group] - starts[group]
        for target in range(starts[group], ends[group]):
            sum_s = 0.0
            sum_s2 = 0.0
            for observer in range(n_disc):
                value = sigma[latent[observer, target]]
                sum_s += value
                sum_s2 += value * value
            sum_target_goodness += sum_s / n_disc
            numerator = (n_disc - 1.0) * sum_s - (sum_s * sum_s - sum_s2)
            denominator = 0.5 * n_disc * (n_disc - 1.0)
            sum_target_disagreement += numerator / denominator
        accumulator[group] += sum_target_goodness / n_targets
        accumulator[3 + group] += sum_target_disagreement / n_targets
    b_index = k1 + 1
    warning_count = 0
    suspicious_count = 0
    for observer in range(latent.shape[0]):
        for target in range(latent.shape[1]):
            state = latent[observer, target]
            if 1 <= state < b_index:
                warning_count += 1
            elif state > b_index:
                suspicious_count += 1
    n_records = latent.shape[0] * latent.shape[1]
    accumulator[6] += warning_count / n_records
    accumulator[7] += suspicious_count / n_records

@njit(cache=False)
def _simulate_replicate(n_disc: int, n_allc: int, n_alld: int, p_norm: float, q_norm: float, e1: float, e2: float, k1: int, sigma: np.ndarray, positive: np.ndarray, negative: np.ndarray, total_events: int, burnin_events: int, interval_events: int, use_seed: bool, seed: int) -> Tuple[np.ndarray, int]:
    if use_seed:
        np.random.seed(seed)
    population_size = n_disc + n_allc + n_alld
    latent = np.zeros((n_disc, population_size), dtype=np.int16)
    accumulator = np.zeros(8, dtype=np.float64)
    n_samples = 0
    next_sample = burnin_events
    for event in range(total_events):
        donor = np.random.randint(0, population_size)
        recipient = np.random.randint(0, population_size - 1)
        if recipient >= donor:
            recipient += 1
        observer = np.random.randint(0, n_disc)
        if donor < n_disc:
            intended_cooperation = _binary_draw(sigma[latent[donor, recipient]])
        elif donor < n_disc + n_allc:
            intended_cooperation = 1
        else:
            intended_cooperation = 0
        realized_cooperation = intended_cooperation
        if intended_cooperation == 1 and np.random.random() < e1:
            realized_cooperation = 0
        recipient_good = _binary_draw(sigma[latent[observer, recipient]])
        if realized_cooperation == 1:
            positive_probability = 1.0 if recipient_good == 1 else p_norm
        else:
            positive_probability = 0.0 if recipient_good == 1 else q_norm
        recorded_positive = _binary_draw(positive_probability)
        if np.random.random() < e2:
            recorded_positive = 1 - recorded_positive
        current = latent[observer, donor]
        latent[observer, donor] = positive[current] if recorded_positive == 1 else negative[current]
        completed_events = event + 1
        if completed_events >= next_sample:
            _accumulate_statistics(latent, sigma, n_disc, n_allc, k1, accumulator)
            n_samples += 1
            next_sample += interval_events
    if n_samples > 0:
        accumulator /= n_samples
    return (accumulator, n_samples)

In [3]:
def snapshot_parameters(norm, regime):
    parameters = dict(MATRIX_PARAMETERS, norm=norm, regime=regime)
    k1, k2 = MULTISTEP_DEPTHS[norm] if regime == "multi_step" else (1, 1)
    parameters.update(k1=k1, k2=k2, alpha=0. if regime == "without_inertia" else 1.,
                      beta=0. if regime == "without_inertia" else 1.)
    return parameters


def simulate_snapshot(parameters):
    p = parameters
    sigma = build_sigma(p["alpha"], p["beta"], p["k1"], p["k2"])
    positive, negative = build_reset_transitions(p["k1"], p["k2"])
    n_records = p["n_disc"] * (p["n_disc"] + p["n_allc"] + p["n_alld"])
    latent = _simulate_condition(p["n_disc"], p["n_allc"], p["n_alld"], *NORM_PARAMETERS[p["norm"]],
        p["e1"], p["e2"], sigma, positive, negative, int(round(p["sweeps"] * n_records)), True, p["seed"])
    return latent, sigma[latent]


def save_figure(figure, name):
    for extension in ("pdf", "svg"):
        figure.savefig(FIGURES / f"{name}.{extension}", bbox_inches="tight", pad_inches=.04)


for k1 in range(7):
    for k2 in range(7):
        positive, negative = build_reset_transitions(k1, k2)
        bad = k1 + 1
        assert np.all(positive[1:bad] == 0)
        assert np.all(negative[bad + 1:] == bad)
        state = 0
        for _ in range(k1 + 1):
            state = negative[state]
        assert state == bad
        for _ in range(k2 + 1):
            state = positive[state]
        assert state == 0


## Reputation matrices

The three target blocks are DISC, ALLC, and ALLD. Each snapshot uses 30 individuals per strategy, 1,000 sweeps, and seed 2027. The multi-step depths are SC (2,0), SH (3,0), SJ (4,3), and SS (1,2).

Without inertia uses $\alpha=\beta=0$; one-step and multi-step inertia use $\alpha=\beta=1$.

In [4]:
matrix_configuration = [snapshot_parameters(norm, regime) for norm in NORMS for regime in REGIMES]
configuration_file = DATA / "snapshot_config.json"
if RECOMPUTE_MATRICES:
    for parameters in matrix_configuration:
        latent, expressed = simulate_snapshot(parameters)
        np.savez_compressed(DATA / f"{parameters['norm']}_{parameters['regime']}.npz", latent=latent, expressed=expressed)
    configuration_file.write_text(json.dumps(matrix_configuration, indent=2))
else:
    if not configuration_file.is_file():
        raise FileNotFoundError("Snapshot data are missing. Supply data/figure2 or set RECOMPUTE_MATRICES=True.")
    if json.loads(configuration_file.read_text()) != matrix_configuration:
        raise ValueError("Snapshot parameters differ from saved data. Set RECOMPUTE_MATRICES=True to regenerate.")

matrices = {}
for parameters in matrix_configuration:
    key = (parameters["norm"], parameters["regime"])
    with np.load(DATA / f"{key[0]}_{key[1]}.npz") as saved:
        latent = saved["latent"]
        expressed = saved["expressed"]
    expected_shape = (parameters["n_disc"], sum(parameters[name] for name in ("n_disc", "n_allc", "n_alld")))
    assert latent.shape == expressed.shape == expected_shape
    assert np.array_equal(expressed, build_sigma(parameters["alpha"], parameters["beta"], parameters["k1"], parameters["k2"])[latent])
    matrices[key] = expressed
print("Loaded 12 reputation matrices.")


Loaded 12 reputation matrices.


In [5]:
BINARY_CMAP = ListedColormap(["#FFFFFF", "#4C78A8"])


def draw_matrix(axis, values, show_observers=False):
    n_observers, total = values.shape
    n_targets = total // 3
    for block, target in enumerate(TARGETS):
        start = block * (n_targets + 2)
        axis.imshow(values[:, block * n_targets:(block + 1) * n_targets], cmap=BINARY_CMAP, vmin=0, vmax=1,
            interpolation="nearest", origin="upper", extent=(start, start + n_targets, n_observers, 0))
        axis.add_patch(Rectangle((start, 0), n_targets, n_observers, fill=False, edgecolor="#777777", linewidth=.45))
    axis.set_xlim(0, 3 * n_targets + 4)
    axis.set_ylim(n_observers, 0)
    axis.set_aspect("equal")
    axis.set_xticks([n_targets / 2 + i * (n_targets + 2) for i in range(3)], TARGETS, fontsize=8)
    axis.tick_params(axis="x", length=0, pad=3)
    axis.set_yticks([])
    if show_observers:
        axis.set_ylabel("DISC observers", fontsize=8)
    for spine in axis.spines.values():
        spine.set_visible(False)


for norm in NORMS:
    for regime in REGIMES:
        figure, axis = plt.subplots(figsize=(5.4, 2.05))
        draw_matrix(axis, matrices[norm, regime], show_observers=True)
        depth = MULTISTEP_DEPTHS[norm] if regime == "multi_step" else (1, 1)
        label = f"{NORM_NAMES[norm]}: {REGIME_NAMES[regime]}"
        if regime == "multi_step":
            label += f" ({depth[0]}, {depth[1]})"
        axis.set_title(label, fontsize=10)
        save_figure(figure, f"matrix_{norm}_{regime}")
        plt.close(figure)
print("Saved 12 matrix panels as PDF and SVG.")


Saved 12 matrix panels as PDF and SVG.


## Depth plane

The scan uses full readout ($\alpha=\beta=1$), 50 individuals per strategy, depths 0-6, one replicate per condition, and seed 2027. Sampling covers sweeps 800-1,000 every five sweeps. Color encodes separation $\Delta g=(g_{\mathrm{DISC}}+g_{\mathrm{ALLC}})/2-g_{\mathrm{ALLD}}$; circles encode observer disagreement. A star marks the smallest depth satisfying $\Delta g\geq0.8$ and $\bar d\leq0.1$.

In [6]:
def summarize_depth(raw):
    rows = []
    for key, group in raw.groupby(["norm", "norm_name", "k1", "k2"], sort=False):
        row = dict(zip(("norm", "norm_name", "k1", "k2"), key))
        row["n_reps"] = len(group)
        for metric in SUMMARY_METRICS:
            values = group[metric].to_numpy(float)
            row[f"{metric}_mean"] = values.mean()
            row[f"{metric}_sd"] = values.std(ddof=1) if len(values) > 1 else np.nan
            row[f"{metric}_sem"] = row[f"{metric}_sd"] / np.sqrt(len(values))
        rows.append(row)
    return pd.DataFrame(rows)


def simulate_depth_scan(parameters):
    p = parameters
    n_records = p["n_disc"] * (p["n_disc"] + p["n_allc"] + p["n_alld"])
    total = int(round(p["sweeps_total"] * n_records))
    burnin = int(round(p["sweeps_burnin"] * n_records))
    interval = int(round(p["sample_interval_sweeps"] * n_records))
    assert 0 <= burnin < total and interval > 0 and p["n_reps"] > 0
    rows = []
    for norm in p["norms"]:
        for k2 in p["strictness_depths"]:
            for k1 in p["tolerance_depths"]:
                sigma = build_sigma(1., 1., k1, k2)
                positive, negative = build_reset_transitions(k1, k2)
                for replicate in range(p["n_reps"]):
                    seed = int(np.random.SeedSequence([p["seed"], NORMS.index(norm), k1, k2, replicate]).generate_state(1, dtype=np.uint32)[0])
                    values, n_samples = _simulate_replicate(p["n_disc"], p["n_allc"], p["n_alld"],
                        *NORM_PARAMETERS[norm], p["e1"], p["e2"], k1, sigma, positive, negative,
                        total, burnin, interval, True, seed)
                    row = dict(norm=norm, norm_name=NORM_NAMES[norm], k1=k1, k2=k2,
                        tolerance_path_length=k1 + 1, strictness_path_length=k2 + 1,
                        replicate=replicate, transition_rule="direct_reset")
                    row.update(dict(zip(METRICS, values)))
                    row.update(mean_disagreement=np.average(values[3:6], weights=[p["n_disc"], p["n_allc"], p["n_alld"]]),
                        cooperator_defector_separation=.5 * (values[0] + values[1]) - values[2],
                        intermediate_fraction=values[6] + values[7], n_samples=n_samples, simulation_seed=seed)
                    rows.append(row)
        print(f"Completed {norm} depth scan.")
    return pd.DataFrame(rows)


raw_file = DATA / "path_length_plane_raw.csv"
summary_file = DATA / "path_length_plane_summary.csv"
config_file = DATA / "path_length_plane_config.json"
if RECOMPUTE_DEPTH_SCAN:
    depth_raw = simulate_depth_scan(DEPTH_PARAMETERS)
    depth_summary = summarize_depth(depth_raw)
    depth_raw.to_csv(raw_file, index=False)
    depth_summary.to_csv(summary_file, index=False)
    config_file.write_text(json.dumps(DEPTH_PARAMETERS, indent=2))
else:
    if not all(path.is_file() for path in (raw_file, summary_file, config_file)):
        raise FileNotFoundError("Depth-scan data are missing. Supply data/figure2 or set RECOMPUTE_DEPTH_SCAN=True.")
    saved_parameters = json.loads(config_file.read_text())
    if any(saved_parameters.get(key) != value for key, value in DEPTH_PARAMETERS.items()):
        raise ValueError("Depth parameters differ from saved data. Set RECOMPUTE_DEPTH_SCAN=True to regenerate.")
    depth_raw = pd.read_csv(raw_file)
    depth_summary = pd.read_csv(summary_file)
expected_conditions = len(DEPTH_PARAMETERS["norms"]) * len(DEPTH_PARAMETERS["tolerance_depths"]) * len(DEPTH_PARAMETERS["strictness_depths"])
assert len(depth_summary) == expected_conditions
assert len(depth_raw) == expected_conditions * DEPTH_PARAMETERS["n_reps"]
assert (depth_summary["n_reps"] == DEPTH_PARAMETERS["n_reps"]).all()
assert not depth_summary.duplicated(["norm", "k1", "k2"]).any()
print(f"Loaded {len(depth_raw)} depth-scan replicates and {len(depth_summary)} conditions.")


Loaded 196 depth-scan replicates and 196 conditions.


In [7]:
SEPARATION_CMAP = LinearSegmentedColormap.from_list("separation", ["#C65A72", "#F7F7F7", "#2166AC"])
SEPARATION_NORM = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)


def bubble_area(disagreement):
    return 420. * np.square(np.clip(np.asarray(disagreement) / .52, 0., 1.))


def draw_depth(axis, norm, title=True):
    frame = depth_summary.loc[depth_summary.norm == norm]
    xvalues, yvalues = DEPTH_PARAMETERS["tolerance_depths"], DEPTH_PARAMETERS["strictness_depths"]
    separation = frame.pivot(index="k2", columns="k1", values="cooperator_defector_separation_mean").reindex(index=yvalues, columns=xvalues).to_numpy()
    disagreement = frame.pivot(index="k2", columns="k1", values="mean_disagreement_mean").reindex(index=yvalues, columns=xvalues).to_numpy()
    assert np.isfinite(separation).all() and np.isfinite(disagreement).all()
    image = axis.pcolormesh(np.arange(len(xvalues) + 1) - .5, np.arange(len(yvalues) + 1) - .5,
        separation, cmap=SEPARATION_CMAP, norm=SEPARATION_NORM, edgecolors="white", linewidth=.4, shading="flat")
    xx, yy = np.meshgrid(np.arange(len(xvalues)), np.arange(len(yvalues)))
    axis.scatter(xx.ravel(), yy.ravel(), s=bubble_area(disagreement).ravel(), facecolor="white",
        edgecolor="#222222", linewidth=.45, alpha=.85, zorder=3)
    for k1, k2, color in ((0, 0, "#222222"), (1, 1, "#C44E52")):
        if k1 in xvalues and k2 in yvalues:
            axis.add_patch(Rectangle((xvalues.index(k1) - .46, yvalues.index(k2) - .46), .92, .92,
                fill=False, edgecolor=color, linewidth=1.35, zorder=4))
    eligible = frame.loc[(frame.cooperator_defector_separation_mean >= .8) & (frame.mean_disagreement_mean <= .1)].copy()
    if len(eligible):
        eligible["K"] = eligible.k1 + eligible.k2
        selected = eligible.sort_values(["K", "cooperator_defector_separation_mean", "mean_disagreement_mean", "k1", "k2"],
            ascending=[True, False, True, True, True]).iloc[0]
        axis.scatter(xvalues.index(int(selected.k1)), yvalues.index(int(selected.k2)), marker="*", s=145,
            facecolor="#FFD34E", edgecolor="#111111", linewidth=.7, zorder=5)
    axis.set_xticks(np.arange(len(xvalues)), xvalues)
    axis.set_yticks(np.arange(len(yvalues)), yvalues)
    axis.set(xlabel=r"Tolerance depth $k_1$", ylabel=r"Strictness depth $k_2$", aspect="equal")
    axis.tick_params(length=3, width=.7)
    if title:
        axis.set_title(NORM_NAMES[norm], fontsize=10)
    return image


for norm in NORMS:
    figure, axis = plt.subplots(figsize=(4.35, 3.65))
    image = draw_depth(axis, norm)
    figure.colorbar(image, ax=axis, fraction=.045, pad=.04, label=r"Separation $\Delta g$", ticks=[-1, 0, 1])
    figure.tight_layout()
    save_figure(figure, f"depth_{norm}")
    plt.close(figure)
print("Saved four depth panels as PDF and SVG.")


Saved four depth panels as PDF and SVG.


## Combined figure

In [8]:
figure = plt.figure(figsize=(15.2, 9.6))
grid = figure.add_gridspec(4, 4, height_ratios=[1.05, 1.05, 1.05, 3.5],
    left=.075, right=.985, bottom=.17, top=.94, hspace=.40, wspace=.26)
for row, regime in enumerate(REGIMES):
    for column, norm in enumerate(NORMS):
        axis = figure.add_subplot(grid[row, column])
        draw_matrix(axis, matrices[norm, regime])
        if row == 0:
            axis.set_title(f"{chr(97 + column)}  {NORM_NAMES[norm]}", loc="left", fontsize=11, fontweight="bold", pad=6)
        if regime == "multi_step":
            k1, k2 = MULTISTEP_DEPTHS[norm]
            axis.text(.5, 1.04, f"({k1}, {k2})", transform=axis.transAxes, ha="center", fontsize=8)
        if column == 0:
            box = axis.get_position()
            figure.text(.012, (box.y0 + box.y1) / 2, REGIME_NAMES[regime], rotation=90,
                va="center", ha="center", fontsize=10)
for column, norm in enumerate(NORMS):
    axis = figure.add_subplot(grid[3, column])
    image = draw_depth(axis, norm, title=False)
    axis.set_title(chr(101 + column), loc="left", fontsize=11, fontweight="bold", pad=6)
    if column:
        axis.set_ylabel("")

reference_handles = [Patch(facecolor="none", edgecolor="#222222", label="Without inertia (0, 0)"),
    Patch(facecolor="none", edgecolor="#C44E52", label="One-step inertia (1, 1)"),
    Line2D([], [], marker="*", linestyle="none", markerfacecolor="#FFD34E", markeredgecolor="#111111",
        markersize=10, label=r"Shortest pair: $\Delta g\geq0.8$, $\bar d\leq0.1$")]
figure.legend(handles=reference_handles, loc="lower center", bbox_to_anchor=(.53, .104), ncol=3, frameon=False, fontsize=9)
color_axis = figure.add_axes([.18, .055, .24, .017])
figure.colorbar(image, cax=color_axis, orientation="horizontal", ticks=[-1, 0, 1], label=r"Separation $\Delta g$")
bubble_handles = [plt.scatter([], [], s=float(bubble_area(value)), facecolor="white", edgecolor="#222222", linewidth=.45,
    label=f"{value:.1f}") for value in (.1, .3, .5)]
figure.legend(handles=bubble_handles, title=r"Observer disagreement $\bar d$", loc="lower center", bbox_to_anchor=(.66, .013),
    ncol=3, frameon=False, columnspacing=1.8, labelspacing=.7, handletextpad=.8)
figure.legend(handles=[Patch(facecolor="#4C78A8", label="Good"), Patch(facecolor="white", edgecolor="#777777", label="Bad")],
    loc="upper right", bbox_to_anchor=(.986, .995), ncol=2, frameon=False, fontsize=9)
save_figure(figure, "Figure2_combined")
plt.show()

expected_names = [f"matrix_{norm}_{regime}" for norm in NORMS for regime in REGIMES] + [f"depth_{norm}" for norm in NORMS] + ["Figure2_combined"]
assert all((FIGURES / f"{name}.{extension}").is_file() for name in expected_names for extension in ("pdf", "svg"))
print("Saved 17 figures, each in PDF and SVG format.")


Saved 17 figures, each in PDF and SVG format.


/var/folders/mq/sp090wk97355w4h_9rtbdswr0000gn/T/ipykernel_41104/3554255172.py:38: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
